<a href="https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2008%20-%20Gradient%20Descent%3A%20Teaching%20a%20Model%20to%20Improve/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 08 — Gradient Descent: Teaching a Model to Improve · Laboratory

↩ **Theory:** [`blog.md`](<blog.md>) — read the lecture first. This notebook tests it.

## Step 1 — The Problem

Chapter 7 gave us a compass: at $(0,0)$ the gradient is $[-38.5, -13.0]$, so both dials
should increase. But *by how much*?

Step the full length of the gradient and you land at $(38.5, 13)$ — about forty units from
an answer that was two units away. A compass is not a journey.

## Step 2 — Prediction

Commit before running. You check these in Step 10.

1. The minimum is where $\nabla L = 0$. Why not just solve that directly and skip iterating?
   Name one reason it fails for real models.
2. With $\eta = 0.05$ from $(0,0)$: how many steps to get the loss below $0.01$?
3. After 200 steps at $\eta=0.05$, will **both** dials have arrived, or just one?
4. The Hessian's largest eigenvalue is $18.3501$. Predict the exact learning rate at which
   training explodes.

In [ ]:
# Step 3 — Intuition: the full-gradient step overshoots wildly.
import numpy as np
np.random.seed(0)

def L(w1, b):
    u, v = w1 - 2, b - 1
    return 8.25 * u**2 + 5.5 * u * v + v**2      # Chapter 7's closed form

def grad(w1, b):
    return np.array([16.5 * (w1 - 2) + 5.5 * (b - 1),
                     5.5 * (w1 - 2) + 2.0 * (b - 1)])

start = np.array([0.0, 0.0])
g = grad(*start)
print("gradient at (0,0):", g, "   loss:", L(*start))
assert np.array_equal(g, [-38.5, -13.0])
assert np.isclose(L(0, 0), 45.0)

full_step = start - g                    # walk the WHOLE gradient
print(f"\nfull-gradient step lands at {full_step}, loss = {L(*full_step):,.1f}")
assert np.isclose(L(*full_step), 13544.0625)
print("The true answer is (2, 1). We aimed downhill and ended 300x higher.")

## Step 4 — The Mathematics Under Test

$$\theta_{t+1} = \theta_t - \eta\nabla L(\theta_t)
\qquad
\mathbf{e}_{t+1} = (I - \eta H)\mathbf{e}_t
\qquad
e_t = (1-\eta\lambda)^t e_0$$

$$\eta < \frac{2}{\lambda_{\max}}
\qquad
\kappa = \frac{\lambda_{\max}}{\lambda_{\min}}
\qquad
\rho = \frac{\kappa-1}{\kappa+1}$$

In [ ]:
# Step 5 — Manual calculation: the trajectory in blog section 4, reproduced exactly.
def descend(eta, steps, start=(0.0, 0.0), record=None):
    theta = np.array(start)
    out = {0: theta.copy()}
    for t in range(1, steps + 1):
        theta = theta - eta * grad(*theta)
        if not np.all(np.isfinite(theta)) or np.max(np.abs(theta)) > 1e10:
            return None, out
        if record and t in record:
            out[t] = theta.copy()
    out['final'] = theta
    return theta, out

marks = {1, 2, 3, 5, 10, 20, 220}
_, traj = descend(0.05, 220, record=marks)

print(f"{'step':>5} {'w1':>9} {'b':>9} {'L':>12}")
for t in [0, 1, 2, 3, 5, 10, 20, 220]:
    w1, b = traj[t]
    print(f"{t:>5} {w1:>9.4f} {b:>9.4f} {L(w1, b):>12.6f}")

# the exact values the lecture prints
assert np.allclose(traj[1], [1.9250, 0.6500], atol=1e-4)
assert np.isclose(L(*traj[1]), 0.3133, atol=1e-4)
assert np.isclose(L(*traj[2]), 0.0091, atol=1e-4)
assert np.allclose(traj[220], [2.0189, 0.9438], atol=1e-3)

print("\nTwo steps take the loss from 45 to 0.009 — a factor of 5000.")
print("Two hundred more steps improve it by only another factor of 30.")

In [ ]:
# Step 5b — the tempting shortcut: solve grad L = 0 directly (blog section 3).
H = np.array([[16.5, 5.5],
              [5.5,  2.0]])
# grad L = H (theta - theta*) = 0  ->  theta = theta*  (H is invertible here)
theta_star = np.array([2.0, 1.0])
assert np.allclose(grad(*theta_star), [0.0, 0.0])
assert np.allclose(np.linalg.solve(H, H @ theta_star), theta_star)
print("solving grad L = 0 gives exactly", theta_star, "in one shot")

# ...and here is why it is a dead end. Cost of solving n equations is about n^3.
for n in [2, 10**6, 10**11]:
    print(f"  n = {n:>12,}  ->  ~{float(n)**3:.1e} operations")
print("\nIt also requires the loss to be quadratic. Add one activation function")
print("(Chapter 25) and there is no closed form to solve at all.")

In [ ]:
# Step 6 — First implementation: the contraction argument, checked step by step.
# Theory: e_{t+1} = (I - eta H) e_t, and along an eigendirection the error is
# multiplied by (1 - eta*lambda) every single step.
evals, evecs = np.linalg.eigh(H)
lmin, lmax = evals[0], evals[-1]
print(f"eigenvalues: lambda_min = {lmin:.6f},  lambda_max = {lmax:.6f}")
assert np.isclose(lmax, 18.350137, atol=1e-5)
assert np.isclose(lmin, 0.149863, atol=1e-5)

eta = 0.05
theta = np.array([0.0, 0.0])
err = theta - theta_star
# components of the error along each eigenvector
comps = [evecs.T @ err]
for _ in range(6):
    theta = theta - eta * grad(*theta)
    comps.append(evecs.T @ (theta - theta_star))

print(f"\n{'step':>5} {'flat comp':>14} {'ratio':>9} {'steep comp':>14} {'ratio':>9}")
for t in range(1, 7):
    r_flat = comps[t][0] / comps[t-1][0]
    r_steep = comps[t][1] / comps[t-1][1]
    print(f"{t:>5} {comps[t][0]:>14.6f} {r_flat:>9.5f} {comps[t][1]:>14.3e} {r_steep:>9.5f}")
    assert np.isclose(r_flat, 1 - eta * lmin, atol=1e-6)
    assert np.isclose(r_steep, 1 - eta * lmax, atol=1e-6)

print(f"\npredicted factors: flat {1-eta*lmin:.5f}, steep {1-eta*lmax:.5f}")
print("Every step multiplies each component by exactly its own constant.")

In [ ]:
# Step 7 — Visualization: the path, the two timescales, and the cliff.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) the descent path on the contours
w1s, bs = np.linspace(-0.5, 3.5, 160), np.linspace(-0.5, 2.0, 160)
W1, B = np.meshgrid(w1s, bs)
Z = 8.25*(W1-2)**2 + 5.5*(W1-2)*(B-1) + (B-1)**2
axes[0].contour(W1, B, np.log10(Z + 1e-9), levels=25, alpha=.6)
path = [np.array([0.0, 0.0])]
for _ in range(60):
    path.append(path[-1] - 0.05 * grad(*path[-1]))
path = np.array(path)
axes[0].plot(path[:, 0], path[:, 1], "o-", ms=3, color="crimson", lw=1)
axes[0].plot(2, 1, "*", ms=18, color="darkgreen")
axes[0].set_title("descent path: one big step, then a crawl")
axes[0].set_xlabel("$w_1$"); axes[0].set_ylabel("$b$")

# (b) the two dials on separate timescales
long_path = [np.array([0.0, 0.0])]
for _ in range(400):
    long_path.append(long_path[-1] - 0.05 * grad(*long_path[-1]))
long_path = np.array(long_path)
axes[1].plot(long_path[:, 0], label="$w_1$ → 2")
axes[1].plot(long_path[:, 1], label="$b$ → 1")
axes[1].axhline(2, ls="--", lw=.8, color="grey"); axes[1].axhline(1, ls="--", lw=.8, color="grey")
axes[1].set_title("one dial sprints, the other crawls")
axes[1].set_xlabel("step"); axes[1].legend(fontsize=8)

# (c) final distance vs learning rate — the cliff at 2/lambda_max
etas = np.linspace(0.005, 0.125, 200)
dists = []
for e in etas:
    with np.errstate(over="ignore", invalid="ignore"):
        th, _ = descend(e, 200)
    dists.append(np.nan if th is None else np.linalg.norm(th - theta_star))
axes[2].semilogy(etas, dists)
axes[2].axvline(2 / lmax, color="crimson", ls="--", label=f"$2/\\lambda_{{max}}$ = {2/lmax:.4f}")
axes[2].set_title("distance after 200 steps")
axes[2].set_xlabel("learning rate"); axes[2].legend(fontsize=8)

for ax in axes: ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Step 8 — The experiment: predict the explosion BEFORE running it.
ceiling = 2 / lmax
kappa = lmax / lmin
rho = (kappa - 1) / (kappa + 1)
print(f"condition number kappa = {kappa:.2f}")
print(f"stability ceiling  2/lambda_max = {ceiling:.6f}")
print(f"best possible contraction rho = (k-1)/(k+1) = {rho:.6f}")
print(f"  -> at best, {np.log(0.5)/np.log(rho):.1f} steps to halve the error")

assert np.isclose(ceiling, 0.108991, atol=1e-5)
assert np.isclose(kappa, 122.45, atol=0.01)
assert np.isclose(rho, 0.983799, atol=1e-5)

In [ ]:
# Step 9 — Change exactly one variable: the learning rate.
print(f"{'eta':>9} {'distance to (2,1)':>20}   verdict")
for eta_try in [0.02, 0.05, 0.10, 0.108, 0.109, 0.12]:
    with np.errstate(over="ignore", invalid="ignore"):
        th, _ = descend(eta_try, 200)
    if th is None:
        print(f"{eta_try:>9} {'overflow':>20}   diverged")
    else:
        d = np.linalg.norm(th - theta_star)
        verdict = "converging" if d < 0.5 else "moving away"
        print(f"{eta_try:>9} {d:>20.6f}   {verdict}")

below, _ = descend(0.108, 200)
above, _ = descend(0.109, 200)
assert np.linalg.norm(below - theta_star) < 0.1     # just under the ceiling
assert np.linalg.norm(above - theta_star) > 1.0     # just over it
print(f"\nThe ceiling was predicted at {ceiling:.6f}; behaviour changes between 0.108 and 0.109.")

## Step 10 — Observe

Against your Step 2 predictions:

1. Solving $\nabla L = 0$ works here and gives $(2,1)$ exactly — but it needs the loss to be
   quadratic, and costs about $n^3$. At $n = 10^{11}$ that is $10^{33}$ operations.
2. **Two** steps put the loss below $0.01$ (45 → 0.3133 → 0.0091).
3. Only **one** dial arrives. $w_1$ reaches $2.08$ in two steps and stops; $b$ is still at
   $0.94$ after two hundred, inching towards 1.
4. Predicted $2/18.3501 = 0.108991$ — and the runs change behaviour between $0.108$ and $0.109$.

Step 6 is the one to sit with: each eigen-component is multiplied by *exactly* its own
constant every step, $0.9925$ for the flat direction and $0.0825$ for the steep one. The
theory is not approximately right; it is right to six decimal places.

## Step 11 — Explain

**Why it converges at all.** The error obeys $\mathbf{e}_{t+1} = (I-\eta H)\mathbf{e}_t$, so
after $t$ steps it is $(I-\eta H)^t\mathbf{e}_0$. Chapter 5 told us what repeated application
of a matrix does: decompose along eigenvectors, and each component is multiplied by
$(1-\eta\lambda)$ per step. Step 6 measured exactly that.

**Why the ceiling is $2/\lambda_{\max}$.** Each component shrinks only if
$|1-\eta\lambda| < 1$, i.e. $\eta < 2/\lambda$. All components must shrink, so the steepest
direction binds. One learning rate has to survive the worst case.

**Why it crawls.** The *smallest* eigenvalue decides the waiting time. Here $\lambda_{\min} =
0.1499$, so at $\eta = 0.05$ the flat direction keeps $99.25\%$ of its error every step. That
is the crawl — and it is why $b$ is still moving after 200 steps while $w_1$ finished at step 2.

> The condition number $\kappa = 122$ is the whole story. Even perfectly tuned, the error can
> shrink no faster than $(\kappa-1)/(\kappa+1) = 0.9838$ per step. Chapter 2 met this as
> feature scaling; Chapter 32's optimizers exist to beat this exact bound.

In [ ]:
# Step 12 — Challenges.

# LEVEL 3 (Derive, then verify): the eta minimizing the worst-case contraction is
# eta* = 2/(lmax + lmin). Check that at eta* the two factors are equal and opposite.
eta_star = 2 / (lmax + lmin)
print(f"eta* = {eta_star:.6f}")
print(f"  factor on steep: {1 - eta_star*lmax:+.6f}")
print(f"  factor on flat : {1 - eta_star*lmin:+.6f}")
assert np.isclose(abs(1 - eta_star*lmax), abs(1 - eta_star*lmin))

# But "optimal" means optimal against the WORST starting point. From (0,0) it is not
# the best choice -- check which eta actually lands closest after 200 steps:
best = min([0.05, 0.08, 0.10, eta_star, 0.108],
           key=lambda e: np.linalg.norm(descend(e, 200)[0] - theta_star))
print(f"\nbest from (0,0) after 200 steps: eta = {best:.6f}  (eta* = {eta_star:.6f})")
# Explain the difference using where the initial error lies in eigen-coordinates.

# LEVEL 4 (Investigate): local minima. f(x) = x^4 - 8x^2 has minima at -2 and +2.
def f(x):  return x**4 - 8*x**2
def fp(x): return 4*x**3 - 16*x
for x0 in [-3.0, -0.5, 0.5, 3.0]:
    x = x0
    for _ in range(300):
        x -= 0.01 * fp(x)
    print(f"  start {x0:>5} -> {x:.6f}")
# Where you land depends only on where you started. Now find the starting point
# that lands on the MAXIMUM at x = 0, and explain why it is unstable.

# YOUR CODE HERE


# LEVEL 5 (Design): invent an update that takes big steps along flat directions and
# small ones along steep directions. What would it need to know, what does it cost
# for n parameters, and why is that cost impossible at n = 1e11?

## Step 13 — Reflection

- [ ] I can say why stepping the full gradient overshot by a factor of forty.
- [ ] I can explain why solving $\nabla L = 0$ is a dead end for real models.
- [ ] I watched each eigen-component shrink by exactly $(1-\eta\lambda)$ per step.
- [ ] I predicted the divergence threshold from an eigenvalue before running anything.
- [ ] I can explain why one dial finished in two steps and the other took hundreds.
- [ ] I can say what a condition number of 122 costs in wall-clock time.
- [ ] I know that zero gradient does not mean minimum.

### The question this chapter leaves open

We can now minimize a loss, prove when it converges, and predict when it explodes. **Given a
loss function.**

But we have been minimizing mean squared error since Chapter 1, where it was chosen because
squaring removes signs and punishes big errors — a *choice*, openly admitted at the time and
never revisited. And our four houses sit exactly on a line, which real houses never do. Real
measurements scatter.

➡️ **Next:** [Chapter 09 — Describing Data: Mean, Variance, Distributions](<../Lecture 09 - Describing Data: Mean, Variance, Distributions/blog.md>)